# Raw Waveform Baseline

This notebook builds a simple baseline classifier directly on raw audio waveforms (time-domain samples).

What it does:
- Loads ESC-50 + FSD50K metadata (cleaned CSVs) and audio files
- Creates train/validation/test splits
- Converts each WAV into a fixed-length 3-second waveform at 16 kHz
- Normalizes amplitudes and trains a small dense neural network


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)

# Paths for the cleaned CSV files and audio folders in Google Drive.
DATA_ROOT = Path('/content/drive/MyDrive/ResoNET/Data')
ESC50_CSV = DATA_ROOT / 'Cleaned_ESC50.csv'
ESC50_AUDIO_DIR = DATA_ROOT / 'ESC50_16000'
FSD50K_DEV_CSV = DATA_ROOT / 'FSD50K_dev_cleaned.csv'
FSD50K_EVAL_CSV = DATA_ROOT / 'FSDK50_eval_cleaned.csv'
FSD50K_DEV_AUDIO_DIR = DATA_ROOT / 'FSD50K.dev_audio_16k'
FSD50K_EVAL_AUDIO_DIR = DATA_ROOT / 'FSD50K.eval_audio_16k'

SAMPLE_RATE = 16000
ESC50_SECONDS = 3.0
FSD50K_SECONDS = 3.0
FIXED_LENGTH = int(SAMPLE_RATE * ESC50_SECONDS)
BATCH_SIZE = 32
EPOCHS = 25
AUTOTUNE = tf.data.AUTOTUNE

Mounted at /content/drive


## 1. Load data and define splits

We read the cleaned CSV files, build full paths to each WAV file, and assign each row to `train`, `val`, or `test` based on dataset-specific split columns.

In [ ]:
def load_dataframe(csv_path, audio_dir, label_col, split_col=None, split_value_map=None):
    """Load cleaned CSV and standardize columns."""
    df = pd.read_csv(csv_path)

    # Full wav path + existence filter
    df['path'] = df['fname'].astype(str).apply(lambda name: str(audio_dir / f'{name}.wav'))
    df = df[df['path'].map(os.path.exists)].copy()

    # Split mapping (dataset-specific -> {train,val,test})
    if split_col is not None and split_value_map is not None:
        df['split_group'] = df[split_col].map(split_value_map)
    else:
        df['split_group'] = 'train'

    # Label normalization
    df['label_name'] = df[label_col].astype(str)
    return df[['path', 'label_name', 'split_group']].copy()

# Load + unify ESC-50 and FSD50K metadata
esc50_df = load_dataframe(
    ESC50_CSV,
    ESC50_AUDIO_DIR,
    label_col='category',
    split_col='fold',
    split_value_map={1: 'test', 2: 'val', 3: 'train', 4: 'train', 5: 'train'},
)
fsd50k_dev_df = load_dataframe(
    FSD50K_DEV_CSV,
    FSD50K_DEV_AUDIO_DIR,
    label_col='labels',
    split_col='split',
    split_value_map={'train': 'train', 'val': 'val'},
)
fsd50k_eval_df = load_dataframe(
    FSD50K_EVAL_CSV,
    FSD50K_EVAL_AUDIO_DIR,
    label_col='labels',
)
fsd50k_eval_df['split_group'] = 'test'

combined_df = pd.concat([esc50_df, fsd50k_dev_df, fsd50k_eval_df], ignore_index=True)

# Split tables
train_df = combined_df[combined_df['split_group'] == 'train'].copy()
val_df = combined_df[combined_df['split_group'] == 'val'].copy()
test_df = combined_df[combined_df['split_group'] == 'test'].copy()

# Label -> id mapping
label_names = sorted(combined_df['label_name'].unique())
label_to_index = {label: idx for idx, label in enumerate(label_names)}
index_to_label = {idx: label for label, idx in label_to_index.items()}

train_df['label_id'] = train_df['label_name'].map(label_to_index).astype(np.int64)
val_df['label_id'] = val_df['label_name'].map(label_to_index).astype(np.int64)
test_df['label_id'] = test_df['label_name'].map(label_to_index).astype(np.int64)

num_classes = len(label_names)
print('Train samples:', len(train_df))
print('Validation samples:', len(val_df))
print('Test samples:', len(test_df))
print('Classes:', label_names)

Train samples: 2151
Validation samples: 262
Test samples: 1044
Classes: ['Alarm', 'Doorbell', 'church_bells', 'clock_alarm', 'door_wood_knock', 'glass_breaking', 'siren']


## 2. Build the waveform dataset

Each audio file is decoded into a mono waveform with a fixed length (3 seconds). We then build efficient `tf.data` pipelines for training and evaluation.

In [ ]:
def decode_audio(file_path, label):
    """WAV -> (FIXED_LENGTH,) waveform."""
    audio = tf.io.read_file(file_path)
    waveform, _ = tf.audio.decode_wav(
        audio,
        desired_channels=1,
        desired_samples=FIXED_LENGTH,
    )
    waveform = tf.squeeze(tf.cast(waveform, tf.float32), axis=-1)
    return waveform, tf.cast(label, tf.int32)


def make_dataset(df, training=False):
    """Build tf.data pipeline for a split."""
    paths = df['path'].to_numpy()
    labels = df['label_id'].to_numpy()

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(min(len(df), 2048), seed=42, reshuffle_each_iteration=True)
    dataset = dataset.map(decode_audio, num_parallel_calls=AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return dataset

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

## 3. Define the dense baseline model

We normalize the waveforms using statistics from the training set, then train a small multilayer perceptron (MLP) on the flattened samples.

In [ ]:
# Global waveform normalization (fit on train only)
normalizer = layers.Normalization(axis=None)
normalizer.adapt(train_ds.map(lambda waveform, label: waveform))

inputs = keras.Input(shape=(FIXED_LENGTH,))
x = normalizer(inputs)

# MLP baseline
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = keras.Model(inputs, outputs)
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 48000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ normalization (Normalization)   │ (None, 48000)          │             3 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 256)            │    12,288,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 7)              │           455 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,329,866 (47.03 MB)

 Trainable params: 12,329,863 (47.03 MB)

 Non-trainable params: 3 (16.00 B)

## 4. Train the model

We train for a fixed number of epochs, using early stopping to avoid overfitting and to restore the best validation accuracy weights.

In [ ]:
# Early stopping on val accuracy
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

Epoch 1/25
68/68 ━━━━━━━━━━━━━━━━━━━━ 16s 177ms/step - accuracy: 0.5328 - loss: 1.8127 - val_accuracy: 0.4695 - val_loss: 2.4141
Epoch 2/25
68/68 ━━━━━━━━━━━━━━━━━━━━ 10s 143ms/step - accuracy: 0.6541 - loss: 2.5634 - val_accuracy: 0.5153 - val_loss: 2.3250
Epoch 3/25
68/68 ━━━━━━━━━━━━━━━━━━━━ 9s 125ms/step - accuracy: 0.7308 - loss: 1.9809 - val_accuracy: 0.5229 - val_loss: 2.0777
Epoch 4/25
68/68 ━━━━━━━━━━━━━━━━━━━━ 10s 143ms/step - accuracy: 0.7610 - loss: 1.6087 - val_accuracy: 0.5267 - val_loss: 2.3237
Epoch 5/25
68/68 ━━━━━━━━━━━━━━━━━━━━ 10s 144ms/step - accuracy: 0.7880 - loss: 1.1825 - val_accuracy: 0.5076 - val_loss: 2.4963
Epoch 6/25
68/68 ━━━━━━━━━━━━━━━━━━━━ 9s 125ms/step - accuracy: 0.8308 - loss: 1.0592 - val_accuracy: 0.4847 - val_loss: 2.3990
Epoch 7/25
68/68 ━━━━━━━━━━━━━━━━━━━━ 10s 142ms/step - accuracy: 0.8503 - loss: 0.9960 - val_accuracy: 0.4809 - val_loss: 2.4891
Epoch 8/25
68/68 ━━━━━━━━━━━━━━━━━━━━ 10s 143ms/step - accuracy: 0.8638 - loss: 0.8530 - val_accura

## 5. Evaluate and predict

Finally, we evaluate on the held-out test set and include a small helper to predict the label for a single WAV file.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')


def predict_file(file_path):
    """Single-file prediction."""
    audio = tf.io.read_file(file_path)
    waveform, _ = tf.audio.decode_wav(
        audio,
        desired_channels=1,
        desired_samples=FIXED_LENGTH,
    )
    waveform = tf.squeeze(tf.cast(waveform, tf.float32), axis=-1)
    probabilities = model.predict(waveform[tf.newaxis, ...], verbose=0)[0]
    predicted_index = int(np.argmax(probabilities))
    return index_to_label[predicted_index], probabilities

sample_file = test_df.iloc[0]['path']
predicted_label, probabilities = predict_file(sample_file)
print(sample_file)
print(predicted_label)
print(probabilities)

33/33 ━━━━━━━━━━━━━━━━━━━━ 10s 297ms/step - accuracy: 0.5556 - loss: 2.2095
Test loss: 2.2095
Test accuracy: 0.5556
/content/drive/MyDrive/ResoNET/ESC50_16000/1-101336-A-30.wav
Doorbell
[1.2511144e-04 9.9710268e-01 5.4732169e-08 2.5460909e-06 2.7143932e-03
 5.2277159e-05 2.9260227e-06]
